In [40]:
import pandas as pd
import numpy as np
import mlflow
import dagshub
import optuna
import json
from pathlib import Path
from sklearn.model_selection import train_test_split,cross_val_score,cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer,TransformedTargetRegressor
from sklearn.preprocessing import OneHotEncoder,MinMaxScaler,OrdinalEncoder,PowerTransformer
from sklearn.metrics import mean_absolute_error,r2_score
from sklearn.ensemble import RandomForestRegressor ,StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor

In [3]:
root_dir = Path.cwd().parent
data_dir = root_dir / 'data' / 'interim' / 'urbaneats-cleaned-dataset.csv'

In [4]:
df = pd.read_csv(data_dir)

In [5]:
df.head()

,rider_id,age,ratings,restaurant_latitude,restaurant_longitude,delivery_latitude,delivery_longitude,order_date,weather,traffic,...,city,order_day,order_month,order_day_of_week,is_weekend,order_time_hour,pickup_time_minutes,order_time_of_day,distance,distance_type
0,INDORES13DEL02,37.0,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,sunny,high,...,INDO,19,3,Saturday,1,11.0,15.0,morning,3.025149,short
1,BANGRES18DEL02,34.0,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,stormy,jam,...,BANG,25,3,Friday,0,19.0,5.0,evening,20.183530,very_long
2,BANGRES19DEL01,23.0,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,sandstorms,low,...,BANG,19,3,Saturday,1,8.0,15.0,morning,1.552758,short
3,COIMBRES13DEL02,38.0,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,sunny,medium,...,COIMB,5,4,Tuesday,0,18.0,10.0,evening,7.790401,medium
4,CHENRES12DEL01,32.0,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,cloudy,high,...,CHEN,26,3,Saturday,1,13.0,15.0,afternoon,6.210138,medium


In [6]:
df.shape

(45502, 27)

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
# drop columns not required for model input

columns_to_drop =  ['rider_id',
                    'restaurant_latitude',
                    'restaurant_longitude',
                    'delivery_latitude',
                    'delivery_longitude',
                    'order_date',
                    "order_time_hour",
                    "order_day",
                    "city",
                    "order_day_of_week",
                    "order_month"]

df.drop(columns=columns_to_drop, inplace=True)

df

,age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,time_taken,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
0,37.0,4.9,sunny,high,2,snack,motorcycle,0.0,no,urban,24,1,15.0,morning,3.025149,short
1,34.0,4.5,stormy,jam,2,snack,scooter,1.0,no,metropolitian,33,0,5.0,evening,20.183530,very_long
2,23.0,4.4,sandstorms,low,0,drinks,motorcycle,1.0,no,urban,26,1,15.0,morning,1.552758,short
3,38.0,4.7,sunny,medium,0,buffet,motorcycle,1.0,no,metropolitian,21,0,10.0,evening,7.790401,medium
4,32.0,4.6,cloudy,high,1,snack,scooter,1.0,no,metropolitian,30,1,15.0,afternoon,6.210138,medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45497,30.0,4.8,windy,high,1,meal,motorcycle,0.0,no,metropolitian,32,0,10.0,morning,1.489846,short
45498,21.0,4.6,windy,jam,0,buffet,motorcycle,1.0,no,metropolitian,36,0,15.0,evening,NaN,NaN
45499,30.0,4.9,cloudy,low,1,drinks,scooter,0.0,no,metropolitian,16,0,15.0,night,4.657195,short
45500,20.0,4.7,cloudy,high,0,snack,motorcycle,1.0,no,metropolitian,26,0,5.0,afternoon,6.232393,medium


In [9]:
# check for missing values

df.isna().sum()

age                    1854
ratings                1908
weather                 525
traffic                 510
vehicle_condition         0
type_of_order             0
type_of_vehicle           0
multiple_deliveries     993
festival                228
city_type              1198
time_taken                0
is_weekend                0
pickup_time_minutes    1640
order_time_of_day      2070
distance               3630
distance_type          3630
dtype: int64

In [10]:
dagshub.init(repo_owner='AvanindraBose', repo_name='Urban-Eats-Food-Delivery-Time-Prediction', mlflow=True)

Accessing as AvanindraBose

Initialized MLflow to track repo "AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction"

Repository AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction initialized!

In [11]:
mlflow.set_tracking_uri('https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow')

# Droping Missing Values and then Selecting the Best Stacking Regressor.

In [12]:
temp_df = df.copy().dropna()

In [13]:
temp_df.isna().sum()

age                    0
ratings                0
weather                0
traffic                0
vehicle_condition      0
type_of_order          0
type_of_vehicle        0
multiple_deliveries    0
festival               0
city_type              0
time_taken             0
is_weekend             0
pickup_time_minutes    0
order_time_of_day      0
distance               0
distance_type          0
dtype: int64

In [14]:
temp_df.shape

(37695, 16)

In [15]:
X = temp_df.drop(columns= ['time_taken'])
y = temp_df['time_taken']

In [16]:
X.sample(10)

,age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
37415,24.0,4.7,fog,jam,1,drinks,scooter,1.0,no,urban,1,10.0,evening,10.707860,long
25450,27.0,4.7,fog,low,0,drinks,motorcycle,1.0,no,metropolitian,1,15.0,morning,1.532235,short
8241,31.0,4.3,fog,jam,2,buffet,scooter,2.0,no,metropolitian,1,10.0,evening,10.757109,long
37605,28.0,4.7,fog,jam,2,snack,scooter,1.0,no,metropolitian,0,15.0,evening,4.656902,short
24547,30.0,4.8,windy,medium,2,drinks,motorcycle,0.0,no,metropolitian,1,15.0,evening,4.590772,short
10653,27.0,4.9,sunny,jam,1,snack,motorcycle,1.0,no,urban,0,15.0,night,12.074047,long
39749,29.0,5.0,stormy,jam,1,meal,motorcycle,0.0,no,metropolitian,0,5.0,evening,13.613850,long
26066,28.0,4.8,sunny,low,0,drinks,motorcycle,0.0,no,metropolitian,0,15.0,night,7.549151,medium
10078,26.0,4.6,fog,medium,0,snack,motorcycle,1.0,no,metropolitian,0,15.0,afternoon,7.772039,medium
6564,33.0,4.9,cloudy,medium,1,snack,scooter,0.0,no,metropolitian,0,15.0,evening,9.220500,medium


In [17]:
y.sample(10)

42608    38
6039     22
3688     28
10193    31
11919    40
37583    16
26608    29
5332     26
39644    15
24748    44
Name: time_taken, dtype: int64

In [18]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [19]:
print("The size of train data is",X_train.shape)
print("The shape of test data is",X_test.shape)

The size of train data is (30156, 15)
The shape of test data is (7539, 15)


In [20]:
temp_df.dtypes

age                    float64
ratings                float64
weather                 object
traffic                 object
vehicle_condition        int64
type_of_order           object
type_of_vehicle         object
multiple_deliveries    float64
festival                object
city_type               object
time_taken               int64
is_weekend               int64
pickup_time_minutes    float64
order_time_of_day       object
distance               float64
distance_type           object
dtype: object

In [21]:
num_cols = X_train.select_dtypes(include=np.number).columns.to_list()

In [22]:
num_cols.remove('vehicle_condition')
num_cols.remove('multiple_deliveries')

In [23]:
num_cols

['age', 'ratings', 'is_weekend', 'pickup_time_minutes', 'distance']

In [24]:
X_train.select_dtypes(include=object).columns.to_list()

['weather',
 'traffic',
 'type_of_order',
 'type_of_vehicle',
 'festival',
 'city_type',
 'order_time_of_day',
 'distance_type']

In [25]:
ordinal_cat_cols = ['traffic','distance_type']

nominal_cat_cols = [
    'weather',
    'type_of_order',
    'type_of_vehicle',
    'festival',
    'city_type',
    'order_time_of_day'
]

In [26]:
len(num_cols + nominal_cat_cols + ordinal_cat_cols)

13

In [27]:
# generate order for ordinal encoding

traffic_order = ["low","medium","high","jam"]

distance_type_order = ["short","medium","long","very_long"]

__Testing the Pipeline with Rf and XGB Regressor.__

In [28]:
run_data_xgb = mlflow.get_run('b49702f40cbd4581a53f8b131c566df7')

In [38]:
run_id = 'b49702f40cbd4581a53f8b131c566df7'
artifact_path = 'xgb_best_params.json'
artifact_uri = f"runs:/{run_id}/{artifact_path}"

local_path = mlflow.artifacts.download_artifacts(artifact_uri, dst_path="artifacts")

In [41]:
with open(local_path) as f :
    xgb_params = json.load(f)

In [42]:
xgb_params

{'n_estimators': 603,
 'max_depth': 11,
 'learning_rate': 0.03417913351080966,
 'min_child_weight': 4,
 'gamma': 0.452621135308464,
 'subsample': 0.863443370509037,
 'colsample_bytree': 0.9651511117267577,
 'colsample_bylevel': 0.8881626404741021,
 'colsample_bynode': 0.7870334195129309,
 'reg_alpha': 8.213030683468667e-08,
 'reg_lambda': 0.2436995143218315,
 'max_delta_step': 1,
 'grow_policy': 'lossguide',
 'tree_method': 'hist',
 'random_state': 42,
 'n_jobs': -1}

In [43]:
run_id = 'c9c3f08094bb4b70983f057e293ed81a'
artifact_path = 'rf_best_params.json'
artifact_uri = f"runs:/{run_id}/{artifact_path}"

local_path = mlflow.artifacts.download_artifacts(artifact_uri, dst_path="artifacts")

In [45]:
with open(local_path) as f :
    rf_params = json.load(f)

In [46]:
rf_params

{'n_estimators': 190,
 'max_depth': 15,
 'max_features': None,
 'min_samples_split': 10,
 'min_samples_leaf': 1,
 'max_samples': 0.8689643457612419,
 'random_state': 42,
 'n_jobs': -1}

In [31]:
preprocessor = ColumnTransformer(
    transformers=[
        ('numerical columns',MinMaxScaler(),num_cols),
        ('nominal categorical columns',OneHotEncoder(drop='first',handle_unknown='ignore',sparse_output=False),nominal_cat_cols),
        ('ordinal categorical columns', OrdinalEncoder(categories=[traffic_order,distance_type_order]),ordinal_cat_cols)
    ],remainder='passthrough',n_jobs=-1,force_int_remainder_cols=False,verbose_feature_names_out=False
)

In [56]:
pipeline_xgb = Pipeline([
        ("preprocessor", preprocessor),
        ("model", XGBRegressor(**xgb_params))
    ])

model_pipe_xgb = TransformedTargetRegressor(
        regressor=pipeline_xgb,
        transformer=PowerTransformer()
    )

In [57]:
scores_xgb = cross_validate(
            model_pipe_xgb,
            X_train,
            y_train,
            cv = 5,
            scoring = {
                "mae":"neg_mean_absolute_error",
                "r2":"r2"
            },
            n_jobs = -1,
            return_train_score = True
        )

In [58]:
scores_xgb

{'fit_time': array([2.92773461, 2.87791514, 2.93864655, 3.13102722, 2.88571858]),
 'score_time': array([0.11399436, 0.11899805, 0.10527992, 0.09047031, 0.09526825]),
 'test_mae': array([-3.04336309, -3.01709938, -3.01719928, -3.01385903, -3.02020812]),
 'train_mae': array([-2.65896487, -2.66957784, -2.65948677, -2.65903616, -2.65699673]),
 'test_r2': array([0.83720464, 0.83972001, 0.84028661, 0.83945239, 0.84048808]),
 'train_r2': array([0.87718958, 0.87631941, 0.87734038, 0.87758362, 0.87726247])}

In [50]:
pipeline_rf = Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(**rf_params))
    ])

model_pipe_rf = TransformedTargetRegressor(
        regressor=pipeline_rf,
        transformer=PowerTransformer()
    )

In [62]:
scores_rf = cross_validate(
            model_pipe_rf,
            X_train,
            y_train,
            cv = 5,
            scoring = {
                "mae":"neg_mean_absolute_error",
                "r2":"r2"
            },
            n_jobs = -1,
            return_train_score = True
        )

In [63]:
scores_rf

{'fit_time': array([3.27412701, 7.15435362, 3.42187881, 6.5191884 , 7.22124648]),
 'score_time': array([4.04140854, 0.26145792, 3.89065456, 0.86651325, 0.28832769]),
 'test_mae': array([-3.09503407, -3.07167526, -3.09312435, -3.06932116, -3.06935761]),
 'train_mae': array([-2.29822933, -2.30213694, -2.2996724 , -2.29898091, -2.28660672]),
 'test_r2': array([0.83056292, 0.83353703, 0.83267518, 0.83286794, 0.83471897]),
 'train_r2': array([0.90527397, 0.90514496, 0.90531622, 0.90546566, 0.90613082])}

# Stacking Regressor Should Be BAsed out of Simple Models. Trying LR,KNN,DT.

In [64]:
def build_model(trial):

    meta_model_name = trial.suggest_categorical("model",["LR","KNN","DT"])

    if meta_model_name == "LR":
        meta = LinearRegression()

    elif meta_model_name == "KNN":
        n_neighbors_knn = trial.suggest_int("n_neighbors_knn",1,15)
        weights_knn = trial.suggest_categorical("weights_knn",["uniform","distance"])
        meta = KNeighborsRegressor(n_neighbors=n_neighbors_knn,
                                        weights=weights_knn,n_jobs=-1)

    elif meta_model_name == "DT":
        max_depth_dt = trial.suggest_int("max_depth_dt",1,10)
        min_samples_split_dt = trial.suggest_int("min_samples_split_dt",2,10)
        min_samples_leaf_dt = trial.suggest_int("min_samples_leaf_dt",1,10)
        meta = DecisionTreeRegressor(max_depth=max_depth_dt,
                                        min_samples_split=min_samples_split_dt,
                                        min_samples_leaf=min_samples_leaf_dt,
                                        random_state=42)

    preprocessor = ColumnTransformer(
    transformers=[
        ('numerical columns',MinMaxScaler(),num_cols),
        ('nominal categorical columns',OneHotEncoder(drop='first',handle_unknown='ignore',sparse_output=False),nominal_cat_cols),
        ('ordinal categorical columns', OrdinalEncoder(categories=[traffic_order,distance_type_order]),ordinal_cat_cols)
    ],remainder='passthrough',n_jobs=-1,force_int_remainder_cols=False,verbose_feature_names_out=False
    )

    stack = StackingRegressor(
        estimators=[
            ("xgb", XGBRegressor(**xgb_params)),
            ("rf", RandomForestRegressor(**rf_params))
        ],
        final_estimator= meta,
        cv = 5,
        n_jobs=-1,
        passthrough=False
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", stack)
    ])

    model_pipe = TransformedTargetRegressor(
        regressor=pipeline,
        transformer=PowerTransformer()
    )

    return model_pipe

In [65]:
def objective(trial):
    
    with mlflow.start_run(run_name=f"trial_{trial.number}",nested=True):
        model_pipe = build_model(trial)
        
        scores = cross_validate(
            model_pipe,
            X_train,
            y_train,
            cv = 5,
            scoring = {
                "mae":"neg_mean_absolute_error",
                "r2":"r2"
            },
            n_jobs = -1,
            return_train_score = True
        )

        train_mae = -scores["train_mae"].mean()
        val_mae = -scores["test_mae"].mean()
        val_mae_std = scores["test_mae"].std()
        train_r2 = scores["train_r2"].mean()
        val_r2 = scores["test_r2"].mean()

        mlflow.log_param("model_type",model_pipe.regressor.named_steps['model'].__class__.__name__)
        mlflow.log_param("trial_number", trial.number)
        mlflow.log_params(trial.params)

        mlflow.log_metric("train_mae_mean", train_mae)
        mlflow.log_metric("val_mae_mean", val_mae)
        mlflow.log_metric("val_mae_std", val_mae_std)
        mlflow.log_metric("train_r2_mean", train_r2)
        mlflow.log_metric("val_r2_mean", val_r2)

        for i, score in enumerate(scores["test_mae"]):
            mlflow.log_metric(f"fold_{i}_val_mae", -score)

        for i, score in enumerate(scores["test_r2"]):
            mlflow.log_metric(f"fold_{i}_val_r2", score)

        trial.set_user_attr("val_mae", val_mae)
        trial.set_user_attr("val_r2", val_r2)

        return val_mae


In [66]:
mlflow.set_experiment("Exp 5 : Stacking Regressor Selection and HP Tuning")
study = optuna.create_study(direction="minimize", study_name="Stacking Regressor Selection and HP Tuning")

with mlflow.start_run(run_name= 'Best Stacking Regressor') as parent_run:
    mlflow.set_tag("n_trials",30)
    mlflow.set_tag("cv_folds",5)
    mlflow.set_tag("objective_metric","val_mae")

    study.optimize(objective,n_trials=30)

    best_trial = study.best_trial

    mlflow.log_param("best_trial_number", best_trial.number)
    mlflow.log_param("best_model", best_trial.params["model"])
    
    mlflow.log_metric("best_cv_mae",best_trial.value)

    trials_df = study.trials_dataframe()
    trials_df.to_csv("optuna_trials.csv", index=False)
    mlflow.log_artifact("optuna_trials.csv")

    best_model_pipe = build_model(best_trial)
    best_model_pipe.fit(X_train,y_train)

    y_pred_train = best_model_pipe.predict(X_train)
    y_pred_test = best_model_pipe.predict(X_test)
    
    train_mae = mean_absolute_error(y_train, y_pred_train)
    test_mae = mean_absolute_error(y_test, y_pred_test)

    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)

    mlflow.log_params(best_trial.params)

    mlflow.log_metric("train_mae", train_mae)
    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("train_r2", train_r2)
    mlflow.log_metric("test_r2", test_r2)

    mlflow.sklearn.log_model(best_model_pipe, "model")


[I 2026-06-05 00:43:48,501] A new study created in memory with name: Stacking Regressor Selection and HP Tuning


🏃 View run trial_0 at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/7/runs/cf5aeaf90fc94f80b24be5f94562df09
🧪 View experiment at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/7


[I 2026-06-05 00:46:19,481] Trial 0 finished with value: 3.053492559302989 and parameters: {'model': 'DT', 'max_depth_dt': 4, 'min_samples_split_dt': 7, 'min_samples_leaf_dt': 7}. Best is trial 0 with value: 3.053492559302989.


🏃 View run trial_1 at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/7/runs/c1f431f0840b4fe89d77eceafe44445b
🧪 View experiment at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/7


[I 2026-06-05 00:48:40,927] Trial 1 finished with value: 3.1068604412167664 and parameters: {'model': 'DT', 'max_depth_dt': 10, 'min_samples_split_dt': 3, 'min_samples_leaf_dt': 3}. Best is trial 0 with value: 3.053492559302989.


...

🏃 View run trial_28 at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/7/runs/e9340c4ec1c54128b693f0048b8d73e0
🧪 View experiment at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/7


[I 2026-06-05 01:44:49,018] Trial 28 finished with value: 3.0421384181352864 and parameters: {'model': 'DT', 'max_depth_dt': 7, 'min_samples_split_dt': 10, 'min_samples_leaf_dt': 10}. Best is trial 7 with value: 3.0238586954353854.


🏃 View run trial_29 at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/7/runs/aaf80adc888c436883d7b6b7b143ef0c
🧪 View experiment at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/7


[I 2026-06-05 01:46:20,896] Trial 29 finished with value: 3.1797704623958145 and parameters: {'model': 'KNN', 'n_neighbors_knn': 7, 'weights_knn': 'uniform'}. Best is trial 7 with value: 3.0238586954353854.
c:\Users\avanindra Bose\Urban Eats Delivery Time Prediction\UrbanEats-Delivery-Time-Prediction\.venv\Lib\site-packages\sklearn\compose\_column_transformer.py:978: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(
2026/06/05 01:46:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/05 01:46:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more informati

🏃 View run Best Stacking Regressor at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/7/runs/2c23e070c6e944a689bf3bd8d3908e98
🧪 View experiment at: https://dagshub.com/AvanindraBose/Urban-Eats-Food-Delivery-Time-Prediction.mlflow/#/experiments/7


In [82]:
best_trial.params

{'model': 'LR'}

In [68]:
exp = study.trials_dataframe()

In [69]:
exp.head()

,number,value,datetime_start,datetime_complete,duration,params_max_depth_dt,params_min_samples_leaf_dt,params_min_samples_split_dt,params_model,params_n_neighbors_knn,params_weights_knn,user_attrs_val_mae,user_attrs_val_r2,state
0,0,3.053493,2026-06-05 00:43:50.138500,2026-06-05 00:46:19.477419,0 days 00:02:29.338919,4.0,7.0,7.0,DT,NaN,NaN,3.053493,0.835909,COMPLETE
1,1,3.106860,2026-06-05 00:46:19.486980,2026-06-05 00:48:40.927337,0 days 00:02:21.440357,10.0,3.0,3.0,DT,NaN,NaN,3.106860,0.827746,COMPLETE
2,2,3.134629,2026-06-05 00:48:40.929341,2026-06-05 00:51:06.527389,0 days 00:02:25.598048,NaN,NaN,NaN,KNN,15.0,distance,3.134629,0.825379,COMPLETE
3,3,3.053493,2026-06-05 00:51:06.534453,2026-06-05 00:53:21.529407,0 days 00:02:14.994954,4.0,1.0,5.0,DT,NaN,NaN,3.053493,0.835909,COMPLETE
4,4,4.103891,2026-06-05 00:53:21.537223,2026-06-05 00:55:52.099475,0 days 00:02:30.562252,NaN,NaN,NaN,KNN,1.0,uniform,4.103891,0.685166,COMPLETE


In [90]:
exp['params_model'].value_counts()

params_model
LR     20
DT      5
KNN     5
Name: count, dtype: int64

In [92]:
exp.groupby('params_model')['value'].agg('mean').sort_values(ascending = True)

params_model
LR     3.023859
DT     3.072031
KNN    3.437256
Name: value, dtype: float64

In [93]:
optuna.visualization.plot_optimization_history(study)

In [94]:
optuna.visualization.plot_param_importances(study)

In [95]:
# partial coord plot

optuna.visualization.plot_parallel_coordinate(study,params=["model"])

# From this Experimentation It is clear that Linar Regression is the best meta estimator. Since LR Does not have any Hyper Parameters , so I will not perform Hyper Paramter Tuning.